**Table of contents**<a id='toc0_'></a>    
- 1. [Setup](#toc1_)    
  - 1.1. [Install Dependencies](#toc1_1_)    
  - 1.2. [Import Libraries](#toc1_2_)    
  - 1.3. [Check Runtime / Device](#toc1_3_)    
- 2. [Load Dataset](#toc2_)    
  - 2.1. [Find Project Root](#toc2_1_)    
  - 2.2. [Read Train / Validation Data](#toc2_2_)    
- 3. [Configuration](#toc3_)    
- 4. [Load Qwen Model Locally](#toc4_)    
- 4.5. [Huấn luyện Mô hình Prefix Tuning](#toc4_5_)    
- 5. [Prompting & Inference](#toc5_)    
  - 5.1. [Build Prompt](#toc5_1_)    
  - 5.2. [Generate Summary](#toc5_2_)    
  - 5.3. [Test One Sample](#toc5_3_)    
- 6. [Evaluate Qwen](#toc6_)    
  - 6.1. [Generate Predictions](#toc6_1_)    
  - 6.2. [Compute ROUGE](#toc6_2_)    
  - 6.3. [Optional: Compute BERTScore](#toc6_3_)    
- 7. [Save Outputs](#toc7_)  

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Setup](#toc0_)

Phần này chuẩn bị môi trường chạy notebook: cài dependency, import thư viện và xác nhận thiết bị tính toán. Nên chạy lần lượt từ trên xuống để các biến và thư viện được khởi tạo đúng thứ tự.

## 1.1. <a id='toc1_1_'></a>[Install Dependencies](#toc0_)

Chỉ chạy cell cài đặt khi môi trường hiện tại chưa có đủ thư viện. Tùy chọn `-U` có thể nâng phiên bản package đang có; nếu import lỗi sau khi cài, hãy khởi động lại kernel rồi chạy lại từ phần Import Libraries.

In [1]:
# Run this cell once if your environment does not have the required packages.
# In VS Code, make sure the selected kernel is the project's virtual environment.

import sys

!{sys.executable} -m pip install -q -U     pandas pyarrow tqdm     transformers accelerate sentencepiece peft datasets     evaluate rouge-score bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires goog

## 1.2. <a id='toc1_2_'></a>[Import Libraries](#toc0_)

Các thư viện được chia theo vai trò: xử lý dữ liệu (`pandas`, `datasets`), mô hình và huấn luyện (`torch`, `transformers`, `peft`), đánh giá (`evaluate`) và các tiện ích chuẩn hóa văn bản, đường dẫn, theo dõi tiến trình.

In [2]:
import os
import gc
import time
import inspect
import re
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import PrefixTuningConfig, TaskType, get_peft_model

import evaluate

## 1.3. <a id='toc1_3_'></a>[Check Runtime / Device](#toc0_)

Cell này xác nhận đúng Python kernel, phiên bản PyTorch và khả năng sử dụng CUDA. Prefix Tuning chỉ cập nhật ít tham số hơn full fine-tuning, nhưng việc tải và chạy Qwen 1.5B vẫn phù hợp với GPU hơn CPU.

In [3]:
print("Python executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
else:
    print("No CUDA GPU detected. Qwen inference on CPU will be very slow.")

Python executable: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU name: Tesla T4
Total VRAM: 14.56 GB


# 2. <a id='toc2_'></a>[Load Dataset](#toc0_)

Notebook dùng tập train để huấn luyện Prefix Tuning và lấy một ví dụ few-shot; tập validation được dùng cho sanity check, chọn checkpoint và đánh giá cuối.

## 2.1. <a id='toc2_1_'></a>[Find Project Root](#toc0_)

Đường dẫn gốc được suy ra từ thư mục đang chạy. Nếu notebook được mở trực tiếp trong thư mục `notebooks/`, đường dẫn sẽ lùi lên một cấp để các thư mục `data/` và `outputs/` luôn được tham chiếu nhất quán.

In [4]:
PROJECT_ROOT = Path.cwd()

# If this notebook is opened from the notebooks/ folder, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Current working directory: /kaggle/working
Project root: /kaggle/working


## 2.2. <a id='toc2_2_'></a>[Read Train / Validation Data](#toc0_)

Dữ liệu Kaggle được ưu tiên khi đường dẫn input tồn tại; nếu không, notebook đọc từ `data/` trong project. Hai file parquet phải có các cột được cấu hình ở phần sau, mặc định là `article` và `summary`.

Biến `df` là bản sao của tập validation và được giữ làm đầu vào chung cho các cell inference, evaluation phía dưới.

In [5]:
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/tlimshadoo4115/datasett")
LOCAL_DATA_DIR = PROJECT_ROOT / "data"

# Prefer Kaggle input paths when running on Kaggle; otherwise use the local project data folder.
DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR

TRAIN_DATA_PATH = DATA_DIR / "train-00000-of-00001.parquet"
VALID_DATA_PATH = DATA_DIR / "valid-00000-of-00001.parquet"

missing_paths = [path for path in [TRAIN_DATA_PATH, VALID_DATA_PATH] if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Cannot find required parquet file(s): "
        + ", ".join(str(path) for path in missing_paths)
    )

print("Using train data file:", TRAIN_DATA_PATH)
print("Using validation data file:", VALID_DATA_PATH)

train_df = pd.read_parquet(TRAIN_DATA_PATH)
valid_df = pd.read_parquet(VALID_DATA_PATH)

# Keep df as the validation dataframe so the existing inference/evaluation cells stay unchanged.
df = valid_df.copy()

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Train columns:", train_df.columns.tolist())
print("Validation columns:", valid_df.columns.tolist())
valid_df.head()

Using train data file: /kaggle/input/datasets/tlimshadoo4115/datasett/train-00000-of-00001.parquet
Using validation data file: /kaggle/input/datasets/tlimshadoo4115/datasett/valid-00000-of-00001.parquet
Train shape: (10775, 2)
Validation shape: (1349, 2)
Train columns: ['article', 'summary']
Validation columns: ['article', 'summary']


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...


# 3. <a id='toc3_'></a>[Configuration](#toc0_)

Các tham số quan trọng:

- `max_token_length`: giới hạn tổng được dùng để chia ngân sách giữa prompt và phần tóm tắt sinh mới.
- `max_source_tokens_cap`, `min_source_tokens`, `token_safety_margin`: kiểm soát độ dài bài viết được đưa vào prompt và chừa khoảng an toàn cho template.
- `max_samples`: chỉ giới hạn số mẫu đánh giá cuối, không giới hạn dữ liệu huấn luyện.
- `num_virtual_tokens`: số token prefix có thể học; base model vẫn được đóng băng bởi PEFT.
- `per_device_train_batch_size` kết hợp với `gradient_accumulation_steps` để tạo batch hiệu dụng lớn hơn mà giảm áp lực VRAM.
- `decoding_strategies`: các chiến lược giải mã được so sánh khi đánh giá.

In [6]:
@dataclass
class EvalConfig:
    # Qwen 1.5B is stronger than 0.5B but needs tighter memory settings.
    model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"

    # Dataset columns
    text_col: str = "article"
    summary_col: str = "summary"

    # Inference / token budget settings
    max_token_length: int = 1536
    max_new_tokens: int = 200
    max_source_tokens_cap: int = 1024
    min_source_tokens: int = 128
    token_safety_margin: int = 32
    lead_sentence_ratio: float = 0.7

    # Evaluation settings. Keep small for local testing.
    max_samples: int = 10

    # Deterministic decoding for reproducible evaluation.
    do_sample: bool = False
    num_beams: int = 1
    decoding_strategies: tuple[tuple[str, int], ...] = (("greedy", 1), ("beam2", 2))
    repetition_penalty: float = 1.1

    # Prefix Tuning settings
    use_sanity_check: bool = True
    num_virtual_tokens: int = 45
    prefix_learning_rate: float = 5e-4
    epochs: int = 5
    early_stopping_patience: int = 2
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    logging_steps: int = 20


config = EvalConfig()
print(config)

EvalConfig(model_name='Qwen/Qwen2.5-1.5B-Instruct', text_col='article', summary_col='summary', max_token_length=1536, max_new_tokens=200, max_source_tokens_cap=1024, min_source_tokens=128, token_safety_margin=32, lead_sentence_ratio=0.7, max_samples=10, do_sample=False, num_beams=1, decoding_strategies=(('greedy', 1), ('beam2', 2)), repetition_penalty=1.1, use_sanity_check=True, num_virtual_tokens=45, prefix_learning_rate=0.0005, epochs=5, early_stopping_patience=2, per_device_train_batch_size=1, gradient_accumulation_steps=4, logging_steps=20)


In [7]:
# Keep only required columns and remove missing rows.
# This prevents NameError problems from undefined TEXT_COL/SUMMARY_COL variables.

TEXT_COL = config.text_col
SUMMARY_COL = config.summary_col

required_cols = [TEXT_COL, SUMMARY_COL]


def clean_summary_dataframe(dataframe: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    missing_cols = [col for col in required_cols if col not in dataframe.columns]

    if missing_cols:
        raise ValueError(
            f"Missing columns in {dataset_name}: {missing_cols}. "
            f"Current columns: {dataframe.columns.tolist()}"
        )

    return dataframe[required_cols].dropna().reset_index(drop=True)


train_df = clean_summary_dataframe(train_df, "train dataset")
valid_df = clean_summary_dataframe(valid_df, "validation dataset")

# One train sample is used as an in-prompt few-shot example for both training and inference.
FEWSHOT_ARTICLE = str(train_df.iloc[0][TEXT_COL]).strip()
FEWSHOT_SUMMARY = str(train_df.iloc[0][SUMMARY_COL]).strip()

# Keep df as validation data for the unchanged inference/evaluation cells below.
df = valid_df.copy()

print("Cleaned train shape:", train_df.shape)
print("Cleaned validation shape:", valid_df.shape)
print("Few-shot source: train_df.iloc[0]")
print("Few-shot article chars:", len(FEWSHOT_ARTICLE))
print("Few-shot summary chars:", len(FEWSHOT_SUMMARY))
df.head()

Cleaned train shape: (10775, 2)
Cleaned validation shape: (1349, 2)
Few-shot source: train_df.iloc[0]
Few-shot article chars: 1599
Few-shot summary chars: 350


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...


# 4. <a id='toc4_'></a>[Load Qwen Model Locally](#toc0_)

Tokenizer và base model Qwen Instruct được tải trước, sau đó gắn Prefix Tuning adapter. Trên CUDA, `float16` giúp giảm VRAM; trên CPU dùng `float32` để tránh các vấn đề thường gặp với `float16`.

Prefix Tuning chỉ huấn luyện các tham số prefix được thêm vào, không cập nhật toàn bộ trọng số Qwen. Kết quả từ `model.print_trainable_parameters()` dùng để kiểm tra tỷ lệ tham số thực sự có thể học.

In [8]:
MODEL_NAME = config.model_name

print("Loading tokenizer:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Use float16 on CUDA to reduce VRAM usage.
# Use float32 on CPU because float16 CPU inference may be unstable/slow.
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading model:", MODEL_NAME)
print("Torch dtype:", torch_dtype)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

# If running on CPU, explicitly move model to CPU.
if not torch.cuda.is_available():
    model = model.to("cpu")

model.config.pad_token_id = tokenizer.pad_token_id
print("Base model loaded successfully.")

peft_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=config.num_virtual_tokens,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

model.eval()
print("Prefix tuning model is ready.")

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading model: Qwen/Qwen2.5-1.5B-Instruct
Torch dtype: torch.float16


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully.
trainable params: 645,120 || all params: 1,544,359,424 || trainable%: 0.0418
Prefix tuning model is ready.


# 4.5. <a id='toc4_5_'></a>[Huấn luyện Mô hình Prefix Tuning](#toc0_)

Luồng huấn luyện gồm các bước chính:

1. Chuẩn hóa khoảng trắng và rút gọn bài viết theo ngân sách token. Với văn bản dài, notebook ưu tiên các câu đầu và cuối để giữ cả bối cảnh mở đầu lẫn thông tin kết luận.
2. Tạo prompt theo đúng chat template dùng lúc inference. Nhãn của phần prompt được đặt thành `-100`, vì vậy loss chỉ được tính trên phần summary mục tiêu.
3. Padding từng batch bằng data collator riêng: `input_ids` dùng pad token, `attention_mask` dùng `0`, còn `labels` dùng `-100`.
4. Chạy sanity check trên tập con nhỏ để phát hiện sớm lỗi dữ liệu hoặc bộ nhớ, sau đó huấn luyện chính thức với early stopping và chọn checkpoint có `eval_loss` thấp nhất.

**Lưu ý:** sanity check thực hiện cập nhật thật trên chính adapter đang dùng. Khi `use_sanity_check=True`, quá trình huấn luyện chính thức tiếp tục từ trạng thái adapter sau sanity check, không khởi tạo lại từ đầu.

Checkpoint tốt nhất được nạp lại vào `model` và chỉ adapter Prefix Tuning cùng tokenizer được lưu tại `outputs/qwen_prefix_tuning_best`.

In [9]:
SYSTEM_MESSAGE = (
    "Bạn là trợ lý chuyên tóm tắt văn bản tiếng Việt. "
    "Luôn bám sát bài viết, không thêm thông tin ngoài nguồn."
)

PROMPT_INSTRUCTIONS = """
Bạn là hệ thống tóm tắt văn bản tiếng Việt có căn cứ.

Nhiệm vụ: Viết bản tóm tắt ngắn gọn, tự nhiên và đúng với nội dung bài viết.

Yêu cầu bắt buộc:
- Chỉ sử dụng thông tin xuất hiện rõ ràng trong bài viết.
- Không suy đoán, không thêm kiến thức bên ngoài, không tự tạo tên riêng, địa điểm, ngày tháng hoặc số liệu.
- Giữ đúng tên riêng, địa điểm, ngày tháng và số liệu quan trọng nếu chúng có trong bài.
- Viết 3-5 câu hoàn chỉnh, mạch lạc, không kết thúc giữa câu.
- Ưu tiên diễn đạt lại ngắn gọn, nhưng không làm thay đổi ý nghĩa.
- Chỉ trả về phần tóm tắt, không giải thích thêm.
""".strip()


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def split_vietnamese_sentences(text: str) -> list[str]:
    text = normalize_whitespace(text)
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?…])\s+", text)
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    return sentences or [text]


def count_tokens(text: str) -> int:
    return len(tokenizer(str(text), add_special_tokens=False)["input_ids"])


def trim_to_sentence_boundary(text: str) -> str:
    text = normalize_whitespace(text)
    matches = list(re.finditer(r"[.!?…][\"”’)]*", text))
    if matches:
        return text[:matches[-1].end()].strip()
    return text


def truncate_text_to_token_limit(text: str, token_limit: int) -> str:
    token_limit = max(1, int(token_limit))
    token_ids = tokenizer(
        normalize_whitespace(text),
        add_special_tokens=False,
        truncation=True,
        max_length=token_limit,
    )["input_ids"]
    truncated = tokenizer.decode(token_ids, skip_special_tokens=True).strip()
    return trim_to_sentence_boundary(truncated) or truncated


def select_sentences_with_budget(
    sentences: list[str],
    token_budget: int,
    *,
    from_end: bool = False,
) -> list[str]:
    selected: list[str] = []
    iterable = reversed(sentences) if from_end else sentences

    for sentence in iterable:
        candidate = [sentence] + selected if from_end else selected + [sentence]
        candidate_text = " ".join(candidate)

        if count_tokens(candidate_text) <= token_budget:
            selected = candidate
            continue

        if selected:
            break

        fallback = truncate_text_to_token_limit(sentence, token_budget)
        if fallback:
            selected = [fallback]
        break

    return selected


def build_prompt(article: str) -> str:
    article = normalize_whitespace(article)

    return f"""
{PROMPT_INSTRUCTIONS}

### Ví dụ few-shot
Bài viết mẫu:
{FEWSHOT_ARTICLE}

Tóm tắt mẫu:
{FEWSHOT_SUMMARY}

### Bài viết cần tóm tắt
{article}

### Tóm tắt
""".strip()


def build_messages(article: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": build_prompt(article)},
    ]


def chat_prompt_token_length(article: str) -> int:
    prompt_text = tokenizer.apply_chat_template(
        build_messages(article),
        tokenize=False,
        add_generation_prompt=True,
    )
    return count_tokens(prompt_text)


def get_source_token_budget() -> int:
    max_prompt_length = max(
        config.min_source_tokens,
        config.max_token_length - config.max_new_tokens - config.token_safety_margin,
    )
    prompt_without_article_tokens = chat_prompt_token_length("")
    available_tokens = max_prompt_length - prompt_without_article_tokens

    return max(
        config.min_source_tokens,
        min(config.max_source_tokens_cap, available_tokens),
    )


def prepare_article_for_prompt(article: str) -> str:
    article = normalize_whitespace(article)
    if not article:
        return ""

    source_token_budget = get_source_token_budget()
    if count_tokens(article) <= source_token_budget:
        return article

    sentences = split_vietnamese_sentences(article)
    separator = "\n...\n"
    separator_tokens = count_tokens(separator)

    tail_budget = max(
        config.min_source_tokens // 2,
        int(source_token_budget * (1 - config.lead_sentence_ratio)),
    )
    lead_budget = max(
        config.min_source_tokens // 2,
        source_token_budget - tail_budget - separator_tokens,
    )

    # Keep the final prompt inside budget even after adding the lead-tail separator.
    while lead_budget + tail_budget + separator_tokens > source_token_budget and tail_budget > 32:
        tail_budget -= 16

    lead_sentences = select_sentences_with_budget(sentences, lead_budget, from_end=False)
    tail_sentences = select_sentences_with_budget(sentences, tail_budget, from_end=True)

    if lead_sentences and tail_sentences:
        prepared_article = " ".join(lead_sentences) + separator + " ".join(tail_sentences)
    elif lead_sentences:
        prepared_article = " ".join(lead_sentences)
    elif tail_sentences:
        prepared_article = " ".join(tail_sentences)
    else:
        prepared_article = truncate_text_to_token_limit(article, source_token_budget)

    if count_tokens(prepared_article) > source_token_budget:
        prepared_article = truncate_text_to_token_limit(prepared_article, source_token_budget)

    return prepared_article.strip()


def format_training_example(example):
    article = prepare_article_for_prompt(example[TEXT_COL])
    summary = normalize_whitespace(example[SUMMARY_COL])

    # Qwen Instruct training prompt mirrors generate_qwen_summary input formatting.
    prompt_text = tokenizer.apply_chat_template(
        build_messages(article),
        tokenize=False,
        add_generation_prompt=True,
    )

    max_prompt_length = max(1, config.max_token_length - config.max_new_tokens)
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_prompt_length,
    )["input_ids"]

    target_ids = tokenizer(
        summary,
        add_special_tokens=False,
        truncation=True,
        max_length=max(1, config.max_new_tokens - 1),
    )["input_ids"]

    if tokenizer.eos_token_id is not None:
        target_ids = target_ids[: max(0, config.max_new_tokens - 1)] + [tokenizer.eos_token_id]
    else:
        target_ids = target_ids[: config.max_new_tokens]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


def causal_lm_data_collator(features):
    input_ids = [torch.tensor(feature["input_ids"], dtype=torch.long) for feature in features]
    attention_mask = [torch.tensor(feature["attention_mask"], dtype=torch.long) for feature in features]
    labels = [torch.tensor(feature["labels"], dtype=torch.long) for feature in features]

    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=tokenizer.pad_token_id,
        ),
        "attention_mask": torch.nn.utils.rnn.pad_sequence(
            attention_mask,
            batch_first=True,
            padding_value=0,
        ),
        "labels": torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        ),
    }


raw_train_dataset = Dataset.from_pandas(train_df[required_cols], preserve_index=False)
raw_eval_dataset = Dataset.from_pandas(valid_df[required_cols], preserve_index=False)

if len(raw_train_dataset) == 0:
    raise ValueError("Train dataset is empty after cleaning.")

if len(raw_eval_dataset) == 0:
    raise ValueError("Validation dataset is empty after cleaning.")

print("Source token budget for article:", get_source_token_budget())
print("Few-shot prompt overhead tokens:", chat_prompt_token_length(""))

train_dataset = raw_train_dataset.map(
    format_training_example,
    remove_columns=raw_train_dataset.column_names,
    desc="Tokenizing train data",
)

eval_dataset = raw_eval_dataset.map(
    format_training_example,
    remove_columns=raw_eval_dataset.column_names,
    desc="Tokenizing eval data",
)

print("Train samples:", len(train_dataset))
print("Eval samples:", len(eval_dataset))

Source token budget for article: 482
Few-shot prompt overhead tokens: 822


Tokenizing train data:   0%|          | 0/10775 [00:00<?, ? examples/s]

Tokenizing eval data:   0%|          | 0/1349 [00:00<?, ? examples/s]

Train samples: 10775
Eval samples: 1349


In [10]:
def make_training_args(output_dir, **overrides):
    args = {
        "output_dir": str(output_dir),
        "learning_rate": config.prefix_learning_rate,
        "per_device_train_batch_size": config.per_device_train_batch_size,
        "per_device_eval_batch_size": config.per_device_train_batch_size,
        "gradient_accumulation_steps": config.gradient_accumulation_steps,
        "logging_steps": config.logging_steps,
        "report_to": [],
        "fp16": torch.cuda.is_available(),
        "remove_unused_columns": False,
        "optim": "adamw_torch",
        "save_total_limit": 2,
    }
    args.update(overrides)

    # Compatibility for Transformers versions using eval_strategy instead of evaluation_strategy.
    training_args_params = inspect.signature(TrainingArguments.__init__).parameters
    if "evaluation_strategy" in args and "evaluation_strategy" not in training_args_params:
        if "eval_strategy" in training_args_params:
            args["eval_strategy"] = args.pop("evaluation_strategy")
        else:
            args.pop("evaluation_strategy")

    return TrainingArguments(**args)


def make_trainer(training_args, train_data, eval_data, callbacks=None):
    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_data,
        "eval_dataset": eval_data,
        "data_collator": causal_lm_data_collator,
    }

    if callbacks:
        trainer_kwargs["callbacks"] = callbacks

    trainer_params = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    return Trainer(**trainer_kwargs)


if hasattr(model, "config"):
    model.config.use_cache = False

if config.use_sanity_check:
    sanity_train_dataset = train_dataset.select(range(min(32, len(train_dataset))))
    sanity_eval_dataset = eval_dataset.select(range(min(16, len(eval_dataset))))

    sanity_args = make_training_args(
        PROJECT_ROOT / "outputs" / "qwen_prefix_sanity",
        max_steps=10,
        num_train_epochs=1,
        evaluation_strategy="steps",
        eval_steps=5,
        save_strategy="no",
        load_best_model_at_end=False,
    )

    sanity_trainer = make_trainer(sanity_args, sanity_train_dataset, sanity_eval_dataset)
    sanity_result = sanity_trainer.train()

    sanity_loss_history = [
        {"step": entry.get("step"), "loss": entry.get("loss")}
        for entry in sanity_trainer.state.log_history
        if "loss" in entry
    ]

    print("Sanity check metrics:", sanity_result.metrics)
    print("Sanity check training loss history:", sanity_loss_history)
else:
    print("Skipping sanity check because config.use_sanity_check is False.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
5,No log,11.022486
10,No log,10.796326


Sanity check metrics: {'train_runtime': 36.1977, 'train_samples_per_second': 1.105, 'train_steps_per_second': 0.276, 'total_flos': 442019817977856.0, 'train_loss': 10.803109741210937, 'epoch': 1.25}
Sanity check training loss history: []


In [11]:
checkpoint_dir = PROJECT_ROOT / "outputs" / "qwen_prefix_checkpoints"
best_prefix_dir = PROJECT_ROOT / "outputs" / "qwen_prefix_tuning_best"

training_args = make_training_args(
    checkpoint_dir,
    num_train_epochs=config.epochs,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = make_trainer(
    training_args,
    train_dataset,
    eval_dataset,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=config.early_stopping_patience)
    ],
)
train_result = trainer.train()

# With load_best_model_at_end=True, trainer.model now holds the best validation-loss checkpoint.
model = trainer.model

if hasattr(model, "config"):
    model.config.use_cache = True

model.eval()

best_prefix_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(best_prefix_dir)
tokenizer.save_pretrained(best_prefix_dir)

print("Training metrics:", train_result.metrics)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)
print("Saved best Prefix Tuning adapter to:", best_prefix_dir)

Epoch,Training Loss,Validation Loss
1,2.912899,2.906252
2,1.894471,1.894721
3,1.442591,1.565639
4,1.382389,1.444010
5,1.367875,1.411622


Training metrics: {'train_runtime': 38184.9277, 'train_samples_per_second': 1.411, 'train_steps_per_second': 0.353, 'total_flos': 5.939172136622746e+17, 'train_loss': 2.285282464087585, 'epoch': 5.0}
Best checkpoint: /kaggle/working/outputs/qwen_prefix_checkpoints/checkpoint-13470
Best validation loss: 1.4116222858428955
Saved best Prefix Tuning adapter to: /kaggle/working/outputs/qwen_prefix_tuning_best


# 5. <a id='toc5_'></a>[Prompting & Inference](#toc0_)

Các hàm xây dựng prompt đã được định nghĩa trước bước tokenization để training và inference dùng cùng một định dạng. Điều này tránh chênh lệch giữa dữ liệu mô hình học và dữ liệu mô hình nhận khi sinh tóm tắt.

## 5.1. <a id='toc5_1_'></a>[Build Prompt](#toc0_)

Cell bên dưới chỉ hiển thị bản xem trước của prompt sau khi áp dụng giới hạn token; nó không huấn luyện hoặc sinh kết quả mới.

In [12]:
# Prompt helpers are defined before tokenization in Section 4.5 so training and inference use the same prompt.
prompt_preview = build_prompt(prepare_article_for_prompt(df.iloc[0][TEXT_COL]))
print(prompt_preview[:2500])

Bạn là hệ thống tóm tắt văn bản tiếng Việt có căn cứ.

Nhiệm vụ: Viết bản tóm tắt ngắn gọn, tự nhiên và đúng với nội dung bài viết.

Yêu cầu bắt buộc:
- Chỉ sử dụng thông tin xuất hiện rõ ràng trong bài viết.
- Không suy đoán, không thêm kiến thức bên ngoài, không tự tạo tên riêng, địa điểm, ngày tháng hoặc số liệu.
- Giữ đúng tên riêng, địa điểm, ngày tháng và số liệu quan trọng nếu chúng có trong bài.
- Viết 3-5 câu hoàn chỉnh, mạch lạc, không kết thúc giữa câu.
- Ưu tiên diễn đạt lại ngắn gọn, nhưng không làm thay đổi ý nghĩa.
- Chỉ trả về phần tóm tắt, không giải thích thêm.

### Ví dụ few-shot
Bài viết mẫu:
Gần 20 sự kiện được tổ chức trên toàn thành phố, kéo dài từ 19/4 đến 10/5. Theo Sở Du lịch Hà Nội, ngoài thu hút du khách, loạt sự kiện cũng là các gợi ý dành cho người dân thủ đô không đi chơi xa và muốn tham gia các hoạt động trong ngày. Một số hoạt động tiêu biểu gồm Lễ hội Du lịch Hà Nội 2024 với chủ đề 'Thăng Long - Hà Nội, Thủ đô quyến rũ'; Triển lãm Ngô Quyền - Anh hùng 

## 5.2. <a id='toc5_2_'></a>[Generate Summary](#toc0_)

Hàm sinh tóm tắt áp dụng chat template của Qwen, chuyển tensor sang cùng device với mô hình và chỉ decode các token mới sinh. Tham số `use_adapter` cho phép bật/tắt Prefix Tuning adapter để so sánh với raw base model trong cùng điều kiện prompt và decoding.

In [13]:
def generate_qwen_summary(
    article: str,
    num_beams: int = 1,
    use_adapter: bool = True,
) -> str:
    article = prepare_article_for_prompt(article)

    # Qwen Instruct models expect chat template formatting.
    input_text = tokenizer.apply_chat_template(
        build_messages(article),
        tokenize=False,
        add_generation_prompt=True,
    )

    max_prompt_length = max(1, config.max_token_length - config.max_new_tokens)
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_prompt_length,
    )

    # Move input tensors to the same device as the model.
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    generation_kwargs = {
        "max_new_tokens": config.max_new_tokens,
        "do_sample": config.do_sample,
        "num_beams": num_beams,
        "repetition_penalty": config.repetition_penalty,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if num_beams > 1:
        generation_kwargs["early_stopping"] = True

    with torch.no_grad():
        if use_adapter:
            outputs = model.generate(**inputs, **generation_kwargs)
        else:
            if not hasattr(model, "disable_adapter"):
                raise RuntimeError("The current model does not support disabling its PEFT adapter.")
            with model.disable_adapter():
                outputs = model.generate(**inputs, **generation_kwargs)

    # Only decode newly generated tokens, not the original prompt.
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    summary = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return summary.strip()

## 5.3. <a id='toc5_3_'></a>[Test One Sample](#toc0_)

So sánh định tính trên cùng một bài viết: raw base model với greedy decoding, Prefix Tuned model với greedy decoding và Prefix Tuned model với beam search (`num_beams=2`). Kết quả một mẫu giúp kiểm tra nhanh hành vi, nhưng không thay thế đánh giá tổng hợp ở phần sau.

In [14]:
sample = df.iloc[0]

article = sample[TEXT_COL]
reference = sample[SUMMARY_COL]

# Same article, prompt, and decoding. Only the Prefix Tuning adapter is switched off/on.
prediction_raw_base = generate_qwen_summary(article, num_beams=1, use_adapter=False)
prediction_tuned_greedy = generate_qwen_summary(article, num_beams=1, use_adapter=True)
prediction_tuned_beam2 = generate_qwen_summary(article, num_beams=2, use_adapter=True)

print("ARTICLE:")
print(article[:1000])

print("REFERENCE SUMMARY:")
print(reference)

print("RAW BASE SUMMARY (adapter disabled, greedy):")
print(prediction_raw_base)

print("PREFIX TUNED SUMMARY (adapter enabled, greedy):")
print(prediction_tuned_greedy)

print("PREFIX TUNED SUMMARY (adapter enabled, beam2):")
print(prediction_tuned_beam2)

ARTICLE:
Giải thưởng công bố gần đây bởi World Travel Awards. Đây là năm thứ hai liên tiếp InterContinental Phu Quoc Long Beach Resort được vinh danh ở hạng mục gia đình trên toàn châu Á. Khu nghỉ dưỡng tọa lạc bên biển Phú Quốc, nổi bật với thiết kế lấy cảm hứng từ đại dương. Khuôn viên rộng rãi với nhiều mảng xanh thiên nhiên đậm chất nhiệt đới. Không gian sảnh lễ tân, phòng nghỉ, villa, nhà hàng... đều được chú trọng để tạo sự hài hòa với biển và cây cối. Du khách có thể chọn nghỉ ngơi tại khu phòng khách sạn rộng rãi, tiện nghi, hoặc những căn hộ, phòng suite, biệt thự cao cấp hướng biển. Mỗi không gian được thiết kế dựa trên tinh thần gắn kết các thành viên trong gia đình. Bên cạnh tiện nghi sang trọng, InterContinental Phu Quoc còn hút khách gia đình nhờ loạt trải nghiệm giải trí, thư giãn đa dạng, phù hợp với mọi độ tuổi. Với trẻ nhỏ, khu Planet Trekker là nơi các bé có thể thoải mái vui chơi, học hỏi từ các đầu sách thiếu nhi, những buổi workshop thủ công... Phụ huynh có thể yê

# 6. <a id='toc6_'></a>[Evaluate Qwen](#toc0_)

Đánh giá được thực hiện trên `max_samples` dòng đầu của tập validation và lặp qua từng chiến lược decoding đã cấu hình.

## 6.1. <a id='toc6_1_'></a>[Generate Predictions](#toc0_)

Mỗi bài viết được sinh một prediction cho từng chiến lược decoding. Nếu một lần sinh gặp lỗi, notebook ghi nhận chuỗi rỗng để tiếp tục các mẫu còn lại; các prediction rỗng sẽ bị loại khi tính metric, vì vậy số mẫu hợp lệ giữa các chiến lược có thể khác nhau.

In [15]:
test_df = df.head(config.max_samples).copy()
generation_strategies = list(config.decoding_strategies)

predictions_by_strategy = {strategy_name: [] for strategy_name, _ in generation_strategies}
references = []
articles = []

start_time = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating summaries"):
    article = row[TEXT_COL]
    reference = str(row[SUMMARY_COL])

    articles.append(article)
    references.append(reference)

    for strategy_name, num_beams in generation_strategies:
        try:
            pred = generate_qwen_summary(article, num_beams=num_beams)
        except RuntimeError as e:
            # Common case: CUDA out of memory.
            print(f"RuntimeError for {strategy_name}:", e)
            pred = ""

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
        except Exception as e:
            # Keep evaluation running even if one sample fails.
            print(f"Error for {strategy_name}:", e)
            pred = ""

        predictions_by_strategy[strategy_name].append(pred)

elapsed = time.time() - start_time
total_generations = len(test_df) * len(generation_strategies)
print(f"Generated {total_generations} summaries for {len(test_df)} articles in {elapsed:.2f} seconds")

Generating summaries:   0%|          | 0/10 [00:00<?, ?it/s]

Generated 20 summaries for 10 articles in 169.91 seconds


In [16]:
result_data = {
    "article": articles,
    "reference_summary": references,
}

for strategy_name, _ in generation_strategies:
    result_data[f"qwen_summary_{strategy_name}"] = predictions_by_strategy[strategy_name]

result_df = pd.DataFrame(result_data)
result_df.head()

,article,reference_summary,qwen_summary_greedy,qwen_summary_beam2
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...,InterContinental Phu Quoc Long Beach Resort nằ...,InterContinental Phu Quoc Long Beach Resort là...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...,Việt Nam đứng đứng vị thứ 15 trong danh sách 2...,"Theo tạp chí du lịch Mỹ Condé Nast Traveler, V..."
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ...",Phú Quốc năm thứ ba liên tiếp vào danh sách 'N...,Phú Quốc năm thứ ba liên tiếp vào danh sách 'N...
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...,Kết day Vietnam vừa công bố hợp tác chiến lược...,KKday Vietnam vừa công bố hợp tác chiến lược S...


## 6.2. <a id='toc6_2_'></a>[Compute ROUGE](#toc0_)

ROUGE đo mức độ trùng khớp từ/cụm từ giữa prediction và reference. Notebook tắt stemming và tính metric riêng trên các prediction hợp lệ của từng chiến lược; cần đối chiếu cột `num_samples` khi so sánh điểm nếu trước đó có lỗi inference.

In [17]:
rouge = evaluate.load("rouge")

metric_inputs_by_strategy = {}
rouge_scores_by_strategy = {}
rouge_rows = []

for strategy_name, _ in generation_strategies:
    strategy_predictions = predictions_by_strategy[strategy_name]
    valid_pairs = [
        (pred, ref)
        for pred, ref in zip(strategy_predictions, references)
        if isinstance(pred, str) and pred.strip()
    ]

    if not valid_pairs:
        print(f"No valid predictions found for {strategy_name}. Skipping ROUGE.")
        continue

    valid_predictions, valid_references = zip(*valid_pairs)
    metric_inputs_by_strategy[strategy_name] = {
        "predictions": list(valid_predictions),
        "references": list(valid_references),
    }

    rouge_scores = rouge.compute(
        predictions=list(valid_predictions),
        references=list(valid_references),
        use_stemmer=False,
    )
    rouge_scores_by_strategy[strategy_name] = rouge_scores
    rouge_rows.append({
        "strategy": strategy_name,
        "num_samples": len(valid_predictions),
        **rouge_scores,
    })

if not rouge_rows:
    raise ValueError("No valid predictions found. Please check model inference errors above.")

# Backward-compatible default variables for the first strategy.
default_metric_strategy = generation_strategies[0][0]
if default_metric_strategy in metric_inputs_by_strategy:
    valid_predictions = metric_inputs_by_strategy[default_metric_strategy]["predictions"]
    valid_references = metric_inputs_by_strategy[default_metric_strategy]["references"]
    rouge_scores = rouge_scores_by_strategy[default_metric_strategy]

rouge_scores_df = pd.DataFrame(rouge_rows)
rouge_scores_df

,strategy,num_samples,rouge1,rouge2,rougeL,rougeLsum
0,greedy,10,0.688427,0.430341,0.475601,0.472619
1,beam2,10,0.678822,0.440874,0.448086,0.445331


## 6.3. <a id='toc6_3_'></a>[Optional: Compute BERTScore](#toc0_)

BERTScore đánh giá mức độ tương đồng ngữ nghĩa tốt hơn ROUGE khi prediction diễn đạt khác từ nhưng vẫn giữ đúng ý reference. Đây là metric tùy chọn, thường tải thêm model đánh giá và tốn nhiều thời gian/bộ nhớ hơn ROUGE; cần chạy cell ROUGE trước để tạo các cặp prediction-reference hợp lệ.

In [18]:
# Optional metric. Run this cell if you want semantic similarity score.
# For Vietnamese, lang="vi" is usually acceptable. If it fails, try model_type="xlm-roberta-large".

bertscore = evaluate.load("bertscore")

bert_scores_by_strategy = {}
bert_rows = []

for strategy_name, metric_inputs in metric_inputs_by_strategy.items():
    bert_scores = bertscore.compute(
        predictions=metric_inputs["predictions"],
        references=metric_inputs["references"],
        lang="vi",
    )

    bert_metrics = {
        "bertscore_precision": sum(bert_scores["precision"]) / len(bert_scores["precision"]),
        "bertscore_recall": sum(bert_scores["recall"]) / len(bert_scores["recall"]),
        "bertscore_f1": sum(bert_scores["f1"]) / len(bert_scores["f1"]),
    }
    bert_scores_by_strategy[strategy_name] = bert_metrics
    bert_rows.append({"strategy": strategy_name, **bert_metrics})

# Backward-compatible default variables for the first strategy.
if default_metric_strategy in bert_scores_by_strategy:
    bert_precision = bert_scores_by_strategy[default_metric_strategy]["bertscore_precision"]
    bert_recall = bert_scores_by_strategy[default_metric_strategy]["bertscore_recall"]
    bert_f1 = bert_scores_by_strategy[default_metric_strategy]["bertscore_f1"]

bert_scores_df = pd.DataFrame(bert_rows)
bert_scores_df

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,strategy,bertscore_precision,bertscore_recall,bertscore_f1
0,greedy,0.784387,0.7855,0.784182
1,beam2,0.779814,0.7823,0.780413


# 7. <a id='toc7_'></a>[Save Outputs](#toc0_)

Notebook lưu prediction và metric thành hai file CSV trong `outputs/`. Encoding `utf-8-sig` giúp nội dung tiếng Việt hiển thị ổn định khi mở bằng các phiên bản Excel phổ biến.

In [19]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

output_file = OUTPUT_DIR / f"qwen_predictions_{config.max_samples}_samples.csv"

result_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saved predictions to:", output_file)

Saved predictions to: /kaggle/working/outputs/qwen_predictions_10_samples.csv


In [20]:
# Save metrics as a small CSV file for report writing.
metrics_rows = []

for strategy_name, metric_inputs in metric_inputs_by_strategy.items():
    rouge_scores = rouge_scores_by_strategy.get(strategy_name, {})
    row = {
        "model_name": config.model_name,
        "strategy": strategy_name,
        "num_samples": len(metric_inputs["predictions"]),
        "rouge1": rouge_scores.get("rouge1"),
        "rouge2": rouge_scores.get("rouge2"),
        "rougeL": rouge_scores.get("rougeL"),
        "rougeLsum": rouge_scores.get("rougeLsum"),
    }

    # Add BERTScore if the optional cell was executed.
    if "bert_scores_by_strategy" in globals() and strategy_name in bert_scores_by_strategy:
        row.update(bert_scores_by_strategy[strategy_name])

    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)
metrics_file = OUTPUT_DIR / f"qwen_metrics_{config.max_samples}_samples.csv"

metrics_df.to_csv(metrics_file, index=False, encoding="utf-8-sig")

print("Saved metrics to:", metrics_file)
metrics_df

Saved metrics to: /kaggle/working/outputs/qwen_metrics_10_samples.csv


,model_name,strategy,num_samples,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1
0,Qwen/Qwen2.5-1.5B-Instruct,greedy,10,0.688427,0.430341,0.475601,0.472619,0.784387,0.7855,0.784182
1,Qwen/Qwen2.5-1.5B-Instruct,beam2,10,0.678822,0.440874,0.448086,0.445331,0.779814,0.7823,0.780413
